# Type 1 (object-level existence) intervention: evaluation, COCO vocabulary

In [ ]:
import sys
sys.path.append("..")

import json
import os
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import splice
import torchvision.transforms.v2 as T
from PIL import Image
from torchvision.models.detection import maskrcnn_resnet50_fpn

from utils import obj_enc_aggregation, filter_boxes
from coco_category_vocab import load_category_names, build_coco_category_dictionary
from intervention_image_level import (
    load_checkpoint,
    load_intervention_testset,
    load_gt_categories,
    load_variants,
    load_boundary_type,
    balanced_accuracy,
    positive_variant_accuracy,
    nb_accuracy,
    load_category_id_by_name,
    apply_intervention,
)
from intervention_object_level_existence import (
    xywh_to_xyxy,
    rescale_box,
    load_coco_gt_boxes,
    load_original_image_sizes,
    load_test_objects_raw,
    match_boxes,
    encode_new_objects,
    apply_object_level_intervention,
)

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 250)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "open_clip:ViT-B-32"

splice_model = splice.load(
    model_name, "laion", 10000, l1_penalty=0.25, device=device, return_weights=True
)
instances_json_path = "../datasets/coco/annotations/instances_val2017.json"
category_names = load_category_names(instances_json_path)
splice_model.dictionary = build_coco_category_dictionary(
    splice_model, model_name, category_names, device
).to(device)
name_to_dim = {name: i for i, name in enumerate(category_names)}

category_id_by_name = load_category_id_by_name(instances_json_path)
category_id_to_name = {cid: name for name, cid in category_id_by_name.items()}

RELEVANT_CATEGORIES = {
    "traffic light", "bicycle", "bus", "train", "bottle", "cup", "pizza",
    "cow", "elephant", "sheep", "dog", "car", "bowl", "truck",
    "couch", "chair", "motorcycle", "surfboard", "person",
}
RELEVANT_CATEGORY_IDS = {category_id_by_name[name] for name in RELEVANT_CATEGORIES}

RULE_NAMES = ["Signal and Ride", "Double Serving", "Herd Alone", "Dog or Car", "Three of a Kind",
              "Car Majority", "Empty Seat", "Single Mode Traffic", "Personal Transport", "Surf Trip"]

RULE_RELEVANT_CATEGORY_NAMES = {
    0: {"traffic light", "bus", "train", "bicycle"},
    1: {"bottle", "cup", "pizza"},
    2: {"cow", "elephant", "sheep", "person"},
    3: {"dog", "car"},
    4: {"bowl", "cup"},
    5: {"car", "truck"},
    6: {"couch", "chair", "person"},
    7: {"bicycle", "motorcycle", "car", "bus"},
    8: {"person", "bicycle", "car"},
    9: {"person", "surfboard"},
}
RULE_RELEVANT_CATEGORY_IDS = {
    rule_idx: {category_id_by_name[name] for name in names}
    for rule_idx, names in RULE_RELEVANT_CATEGORY_NAMES.items()
}
assert set().union(*RULE_RELEVANT_CATEGORY_NAMES.values()) == RELEVANT_CATEGORIES

In [ ]:
preprocessor = T.Compose([
    T.Resize(size=224, interpolation=T.InterpolationMode.BICUBIC, max_size=None, antialias=True),
    T.CenterCrop(size=(224, 224)),
    T.Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711)),
])
to_224 = T.Compose([T.Resize((224, 224)), T.ToTensor()])

base_embeddings_path = "../encodings/cocologic/clip/splice_embeddings_0.25_coco_categories.h5"
obj_embeddings_path = "../encodings/cocologic/obj_encodings/clip/obj_7_crop_min_0.01_max_0.85_score_0.2_iou_0.5_model_mask-rcnn_0.25_coco_categories.h5"
laion_obj_embeddings_path = "../encodings/cocologic/obj_encodings/clip/obj_7_crop_min_0.01_max_0.85_score_0.2_iou_0.5_model_mask-rcnn_0.25.h5"
test_json_path = "../data/cocologic/cocologic_test.json"
image_base_path = "../datasets/coco/images"
aggregation = "sum"
num_objects_training = 7
iou_threshold = 0.5
strategy = "greedy"

model = load_checkpoint(f"../checkpoints/cocologic/{aggregation}_super_multilabel_seed42_coco_categories.pt", device=device)

embeddings, obj_embeddings_padded, img_ids, labels = load_intervention_testset(
    base_embeddings_path, obj_embeddings_path, num_objects_training
)
labels = labels.float()
variants = load_variants(test_json_path, img_ids)
boundary_type = load_boundary_type(test_json_path, img_ids)
gt_test = load_gt_categories(test_json_path, img_ids)

raw_embeddings_list, raw_boxes_list, obj_img_ids = load_test_objects_raw(obj_embeddings_path)
id_to_raw_idx = {iid.item(): idx for idx, iid in enumerate(obj_img_ids)}

gt_boxes_by_image = load_coco_gt_boxes(instances_json_path)
orig_sizes = load_original_image_sizes(instances_json_path)
with open(test_json_path) as f:
    meta = json.load(f)

with open(instances_json_path) as f:
    instances = json.load(f)
crowd_boxes_by_image = defaultdict(list)
for ann in instances["annotations"]:
    if ann["iscrowd"] == 1:
        crowd_boxes_by_image[ann["image_id"]].append(xywh_to_xyxy(ann["bbox"]))
del instances

n_images = len(img_ids)
baseline_aggregated = obj_enc_aggregation(aggregation, obj_embeddings_padded, embeddings, only_objects=False)
with torch.no_grad():
    baseline_logits = model(baseline_aggregated.to(device)).cpu()
print(n_images, "test images")

In [ ]:
IMAGE_AREA = 224 * 224
MIN_SIZE, MAX_SIZE, NMS_IOU = 0.01, 0.85, 0.5
EFFECTIVE_MIN_SCORE = 0.5   # default of generate_object_propsals(0.5)
NOMINAL_MIN_SCORE = 0.2     # min_score 


def iou_matrix(boxes_a, boxes_b):
    # same arithmetic as utils.calculate_iou, vectorized in float64
    a = np.asarray(boxes_a, dtype=np.float64).reshape(-1, 4)[:, None, :]
    b = np.asarray(boxes_b, dtype=np.float64).reshape(-1, 4)[None, :, :]
    iw = np.maximum(0, np.minimum(a[..., 2], b[..., 2]) - np.maximum(a[..., 0], b[..., 0]))
    ih = np.maximum(0, np.minimum(a[..., 3], b[..., 3]) - np.maximum(a[..., 1], b[..., 1]))
    inter = iw * ih
    union = (a[..., 2] - a[..., 0]) * (a[..., 3] - a[..., 1]) + (b[..., 2] - b[..., 0]) * (b[..., 3] - b[..., 1]) - inter
    return np.where(union > 0, inter / np.where(union > 0, union, 1), 0.0)


def intersection_over_first(boxes_a, boxes_b):
    a = np.asarray(boxes_a, dtype=np.float64).reshape(-1, 4)[:, None, :]
    b = np.asarray(boxes_b, dtype=np.float64).reshape(-1, 4)[None, :, :]
    iw = np.maximum(0, np.minimum(a[..., 2], b[..., 2]) - np.maximum(a[..., 0], b[..., 0]))
    ih = np.maximum(0, np.minimum(a[..., 3], b[..., 3]) - np.maximum(a[..., 1], b[..., 1]))
    area_a = (a[..., 2] - a[..., 0]) * (a[..., 3] - a[..., 1])
    return np.where(area_a > 0, iw * ih / np.where(area_a > 0, area_a, 1), 0.0)


def greedy_pairs(ious, thr):
    candidates = sorted(
        ((ious[p, g], p, g) for p in range(ious.shape[0]) for g in range(ious.shape[1]) if ious[p, g] >= thr),
        reverse=True,
    )
    pairs, used_p, used_g = [], set(), set()
    for iou, p, g in candidates:
        if p in used_p or g in used_g:
            continue
        pairs.append((p, g, iou))
        used_p.add(p)
        used_g.add(g)
    return pairs


def trace_pipeline(boxes, scores):
    boxes_l, scores_l = boxes.tolist(), scores.tolist()
    rel_area = [(b[2] - b[0]) * (b[3] - b[1]) / IMAGE_AREA for b in boxes_l]
    ious = iou_matrix(boxes_l, boxes_l) if boxes_l else np.zeros((0, 0))
    remaining = torch.argsort(scores, descending=True).tolist()
    selected, suppressed = [], set()
    while remaining:
        cur = remaining[0]
        if rel_area[cur] < MIN_SIZE or rel_area[cur] > MAX_SIZE or scores_l[cur] < EFFECTIVE_MIN_SCORE:
            remaining = remaining[1:]
            continue
        selected.append(cur)
        keep = []
        for idx in remaining[1:]:
            if ious[cur, idx] <= NMS_IOU:
                keep.append(idx)
            else:
                suppressed.add(idx)
        remaining = keep
    retained = [s for s in selected if scores_l[s] > NOMINAL_MIN_SCORE][:num_objects_training]
    retained_set = set(retained)
    fate = []
    for d in range(len(boxes_l)):
        if rel_area[d] < MIN_SIZE or rel_area[d] > MAX_SIZE:
            fate.append("size filter")
        elif scores_l[d] < NOMINAL_MIN_SCORE:
            fate.append("score < 0.2")
        elif scores_l[d] < EFFECTIVE_MIN_SCORE:
            fate.append("score in [0.2, 0.5)")
        elif d in suppressed:
            fate.append("NMS")
        elif d in retained_set:
            fate.append("retained")
        else:
            fate.append("top-k")
    return fate, selected, retained

In [ ]:
det_model = maskrcnn_resnet50_fpn(pretrained=True).eval().to(device)

records = []
intervened_obj_list, after_remove_obj_list = [], []
n_filter_mismatch, n_repro_exact = 0, 0

for i in range(n_images):
    image_id = img_ids[i].item()
    raw_idx = id_to_raw_idx[image_id]
    obj_emb = raw_embeddings_list[raw_idx].to(device)
    obj_boxes = raw_boxes_list[raw_idx].to(device)

    file_name = meta["images"][str(image_id)]["file_name"]
    image_224 = to_224(Image.open(os.path.join(image_base_path, file_name)).convert("RGB")).to(device)

    new_emb, new_boxes, num_removed, num_added, added_ids, kept_ids, after_remove_emb = apply_object_level_intervention(
        obj_emb, obj_boxes, image_224,
        gt_boxes_by_image[image_id], orig_sizes[image_id],
        splice_model, preprocessor, device,
        iou_threshold=iou_threshold, strategy=strategy,
        num_objects_training=num_objects_training,
        relevant_category_ids=RELEVANT_CATEGORY_IDS,
    )
    intervened_obj_list.append(F.pad(new_emb, (0, 0, 0, num_objects_training - new_emb.shape[0])).cpu())
    after_remove_obj_list.append(F.pad(after_remove_emb, (0, 0, 0, num_objects_training - after_remove_emb.shape[0])).cpu())

    gt = gt_boxes_by_image[image_id]
    gt_boxes_224 = [rescale_box(b["bbox_xyxy"], orig_sizes[image_id], (224, 224)) for b in gt]
    gt_cats = [b["category_id"] for b in gt]
    stored_boxes = obj_boxes.cpu()
    matched, unmatched_pred, unmatched_gt = match_boxes(stored_boxes.tolist(), gt_boxes_224, iou_threshold, strategy)

    with torch.no_grad():
        det = det_model([image_224])[0]
    raw_boxes, raw_scores, raw_labels = det["boxes"].cpu(), det["scores"].cpu(), det["labels"].cpu()
    fate, selected, retained = trace_pipeline(raw_boxes, raw_scores)
    utils_selected = [s.item() for s in filter_boxes(raw_boxes, raw_scores, IMAGE_AREA, iou_threshold=NMS_IOU,
                                                     min_size=MIN_SIZE, max_size=MAX_SIZE,
                                                     minimum_score=EFFECTIVE_MIN_SCORE)]
    n_filter_mismatch += utils_selected != selected
    repro = raw_boxes[retained]
    n_repro_exact += repro.shape == stored_boxes.shape and torch.allclose(repro, stored_boxes, atol=0.05)

    stored_labels = [-1] * len(stored_boxes)
    if len(stored_boxes) and len(retained):
        ious_sr = iou_matrix(stored_boxes.tolist(), repro.tolist())
        for s in range(len(stored_boxes)):
            j = int(ious_sr[s].argmax())
            if ious_sr[s, j] >= 0.9:
                stored_labels[s] = int(raw_labels[retained[j]])

    records.append({
        "image_id": image_id,
        "gt_boxes": gt_boxes_224, "gt_cats": gt_cats,
        "crowd_boxes": [rescale_box(b, orig_sizes[image_id], (224, 224)) for b in crowd_boxes_by_image[image_id]],
        "stored_boxes": stored_boxes.tolist(), "stored_labels": stored_labels,
        "matched": matched, "unmatched_pred": unmatched_pred, "unmatched_gt": unmatched_gt,
        "kept_ids": kept_ids, "kept_emb": after_remove_emb.cpu(),
        "added_ids": added_ids, "added_boxes": new_boxes[new_boxes.shape[0] - num_added:].cpu().tolist(),
        "added_emb": new_emb[new_emb.shape[0] - num_added:].cpu(),
        "num_removed": num_removed, "num_added": num_added,
        "raw_boxes": raw_boxes.tolist(), "raw_scores": raw_scores.tolist(), "raw_labels": raw_labels.tolist(),
        "fate": fate,
    })

intervened_obj = torch.stack(intervened_obj_list)
after_remove_obj = torch.stack(after_remove_obj_list)
intervened_aggregated = obj_enc_aggregation(aggregation, intervened_obj, embeddings, only_objects=False)
after_remove_aggregated = obj_enc_aggregation(aggregation, after_remove_obj, embeddings, only_objects=False)
with torch.no_grad():
    intervened_logits = model(intervened_aggregated.to(device)).cpu()
    after_remove_logits = model(after_remove_aggregated.to(device)).cpu()

n_unlabeled = sum(l == -1 for r in records for l in r["stored_labels"])
n_stored = sum(len(r["stored_boxes"]) for r in records)
print(f"trace_pipeline selection == utils.filter_boxes: {n_images - n_filter_mismatch}/{n_images} images")
print(f"Mask R-CNN re-run reproduces the stored boxes exactly (atol 0.05): {n_repro_exact}/{n_images} images")
print(f"stored boxes without a reproduced partner (IoU >= 0.9, no detector class): {n_unlabeled}/{n_stored}")

In [ ]:
def per_rule(logits, mask=None):
    if mask is None:
        mask = torch.ones(len(logits), dtype=torch.bool)
    return {
        "B-Acc": 100 * balanced_accuracy(logits[mask], labels[mask]),
        "PosVar-Acc": 100 * positive_variant_accuracy(logits[mask], variants[mask]),
        "NB-Acc": 100 * nb_accuracy(logits[mask], boundary_type[mask]),
    }

SETTINGS = {"Baseline": baseline_logits, "Remove only": after_remove_logits, "Remove + Add": intervened_logits}
METRICS = ["B-Acc", "PosVar-Acc", "NB-Acc"]
setting_metrics = {name: per_rule(lg) for name, lg in SETTINGS.items()}


per_rule_table = pd.DataFrame(
    {(metric, name): setting_metrics[name][metric].numpy() for metric in METRICS for name in SETTINGS},
    index=RULE_NAMES,
)
for metric in METRICS:
    per_rule_table[(metric, "delta R+A")] = per_rule_table[(metric, "Remove + Add")] - per_rule_table[(metric, "Baseline")]
per_rule_table = per_rule_table[[(m, s) for m in METRICS for s in list(SETTINGS) + ["delta R+A"]]]
print("Per rule (%)")
print(per_rule_table.astype(float).round(1).to_string())

In [ ]:
STAGE_RANK = {"size filter": 1, "score < 0.2": 2, "score in [0.2, 0.5)": 3, "NMS": 4, "top-k": 5, "retained": 6}
STAGE_REASON = {"size filter": "removed by size filter", "score < 0.2": "below score 0.2",
                "score in [0.2, 0.5)": "score in [0.2, 0.5) (default 0.5 threshold)", "NMS": "suppressed by NMS",
                "top-k": "cut by top-k (7)", "retained": "retained, matched to another GT"}
REASON_ORDER = list(STAGE_REASON.values())[::-1] + ["correct class, IoU in [0.1, 0.5)", "wrong class only, IoU in [0.1, 0.5)",
                                                     "no detection (IoU < 0.1)"]

miss_rows = []
for r in records:
    if not r["unmatched_gt"]:
        continue
    raw = r["raw_boxes"]
    ious = iou_matrix(raw, r["gt_boxes"]) if raw else np.zeros((0, len(r["gt_boxes"])))
    for g in r["unmatched_gt"]:
        cat = r["gt_cats"][g]
        box = r["gt_boxes"][g]
        cand = [d for d in range(len(raw)) if ious[d, g] >= 0.5]
        if cand:
            d = max(cand, key=lambda d: (STAGE_RANK[r["fate"][d]], ious[d, g]))
            reason, cls = STAGE_REASON[r["fate"][d]], r["raw_labels"][d] == cat
        else:
            near = [d for d in range(len(raw)) if 0.1 <= ious[d, g] < 0.5]
            near_same = [d for d in near if r["raw_labels"][d] == cat]
            if near_same:
                reason, cls = "correct class, IoU in [0.1, 0.5)", True
            elif near:
                reason, cls = "wrong class only, IoU in [0.1, 0.5)", False
            else:
                reason, cls = "no detection (IoU < 0.1)", None
        miss_rows.append({"image_id": r["image_id"], "category": category_id_to_name[cat],
                          "relevant": cat in RELEVANT_CATEGORY_IDS, "reason": reason,
                          "class": {True: "correct", False: "wrong", None: "-"}[cls],
                          "small GT (< 1 % of image)": (box[2] - box[0]) * (box[3] - box[1]) / IMAGE_AREA < MIN_SIZE})
miss_df = pd.DataFrame(miss_rows)


In [ ]:
SIZE_BUCKETS = ["< 1 %", "1-10 %", "> 10 %"]

def size_bucket(box):
    a = (box[2] - box[0]) * (box[3] - box[1]) / IMAGE_AREA
    return "< 1 %" if a < 0.01 else ("1-10 %" if a <= 0.10 else "> 10 %")

gt_rows = []
for r in records:
    hit = {g for _, g, _ in r["matched"]}
    for g, (box, c) in enumerate(zip(r["gt_boxes"], r["gt_cats"])):
        gt_rows.append({"bucket": size_bucket(box), "relevant": c in RELEVANT_CATEGORY_IDS,
                        "person": category_id_to_name[c] == "person", "matched": g in hit})
gt_df = pd.DataFrame(gt_rows)
assert gt_df["matched"].sum() == sum(len(r["matched"]) for r in records)


In [ ]:
# size bucket of each missed GT object
miss_buckets = [size_bucket(r["gt_boxes"][g]) for r in records if r["unmatched_gt"] for g in r["unmatched_gt"]]
assert len(miss_buckets) == len(miss_df)
miss_b = miss_df.assign(bucket=miss_buckets)
assert ((miss_b["bucket"] == "< 1 %") == miss_b["small GT (< 1 % of image)"]).all()

def reasons_by_bucket(df):
    counts = pd.crosstab(df["reason"], df["bucket"]).reindex(index=REASON_ORDER, columns=SIZE_BUCKETS).fillna(0).astype(int)
    pct = 100 * counts / counts.sum()
    t = pd.concat({"n": counts, "% of bucket": pct.round(1)}, axis=1)
    t.loc["total"] = list(counts.sum()) + [100.0] * len(SIZE_BUCKETS)
    return t

for name, df in [("All categories", miss_b), ("19 rule-relevant categories", miss_b[miss_b["relevant"]]),
                 ("person", miss_b[miss_b["category"] == "person"])]:
    print(name)
    print(reasons_by_bucket(df).to_string())
    print()

## Accuracy


In [ ]:
MIN_ADD_AREA = 0.01

variant_obj_list, variant_rows = [], []
n_same, n_checked = 0, 0
for i, r in enumerate(records):
    kept = r["kept_emb"].to(device)
    free_slots = num_objects_training - kept.shape[0]
    entries = [(r["gt_boxes"][g], r["gt_cats"][g]) for g in r["unmatched_gt"]
               if (r["gt_boxes"][g][2] - r["gt_boxes"][g][0]) * (r["gt_boxes"][g][3] - r["gt_boxes"][g][1]) / IMAGE_AREA >= MIN_ADD_AREA]
    entries.sort(key=lambda e: (e[1] in RELEVANT_CATEGORY_IDS, (e[0][2] - e[0][0]) * (e[0][3] - e[0][1])), reverse=True)
    to_add = entries[:max(free_slots, 0)]
    if to_add:
        file_name = meta["images"][str(r["image_id"])]["file_name"]
        image_224 = to_224(Image.open(os.path.join(image_base_path, file_name)).convert("RGB")).to(device)
        new = encode_new_objects(image_224, [e[0] for e in to_add], splice_model, preprocessor, device)
        emb = torch.cat([kept, new], dim=0)
    else:
        emb = kept
    variant_obj_list.append(F.pad(emb, (0, 0, 0, num_objects_training - emb.shape[0])).cpu())
    added_cats = [e[1] for e in to_add]
    variant_rows.append({"added": len(to_add), "added relevant": sum(c in RELEVANT_CATEGORY_IDS for c in added_cats),
                         "full added": r["num_added"],
                         "full added relevant": sum(c in RELEVANT_CATEGORY_IDS for c in r["added_ids"]),
                         "new relevant category": bool((set(added_cats) & RELEVANT_CATEGORY_IDS) - set(r["kept_ids"]))})
    has_small = any((r["gt_boxes"][g][2] - r["gt_boxes"][g][0]) * (r["gt_boxes"][g][3] - r["gt_boxes"][g][1]) / IMAGE_AREA < MIN_ADD_AREA
                    for g in r["unmatched_gt"])
    if not has_small:
        n_checked += 1
        n_same += torch.allclose(variant_obj_list[-1], intervened_obj_list[i], atol=1e-6)

variant_obj = torch.stack(variant_obj_list)
variant_aggregated = obj_enc_aggregation(aggregation, variant_obj, embeddings, only_objects=False)
with torch.no_grad():
    variant_logits = model(variant_aggregated.to(device)).cpu()
variant_df = pd.DataFrame(variant_rows)
print(f"images without a small unmatched GT object: variant == full Type 1 in {n_same}/{n_checked}")

In [ ]:
SETTINGS_V = {"Baseline": baseline_logits, "Remove only": after_remove_logits,
              "Remove + Add (full)": intervened_logits, "Remove + Add (>= 1 %)": variant_logits}
setting_metrics_v = {name: per_rule(lg) for name, lg in SETTINGS_V.items()}

print("Mean over the 10 rules (%)")
print(pd.DataFrame([{"setting": name, **{k: torch.nanmean(v).item() for k, v in m.items()}}
                    for name, m in setting_metrics_v.items()]).round(2).to_string(index=False))
print()

cols = ["Baseline", "Remove + Add (full)", "Remove + Add (>= 1 %)"]
per_rule_v = pd.DataFrame(
    {(metric, name): setting_metrics_v[name][metric].numpy() for metric in METRICS for name in cols}, index=RULE_NAMES)
for metric in METRICS:
    per_rule_v[(metric, "delta >=1% vs base")] = per_rule_v[(metric, "Remove + Add (>= 1 %)")] - per_rule_v[(metric, "Baseline")]
    per_rule_v[(metric, "delta >=1% vs full")] = per_rule_v[(metric, "Remove + Add (>= 1 %)")] - per_rule_v[(metric, "Remove + Add (full)")]
per_rule_v = per_rule_v[[(m, s) for m in METRICS for s in cols + ["delta >=1% vs base", "delta >=1% vs full"]]]
print("Per rule (%)")
print(per_rule_v.astype(float).round(1).to_string())

## Intervention effect


In [ ]:
def frac(num, den):
    return f"{num} / {den} = {100 * num / den:.1f} %"

summary = []  

n_gt = sum(len(r["gt_boxes"]) for r in records)
n_gt_matched = sum(len(r["matched"]) for r in records)
n_gt_rel = sum(c in RELEVANT_CATEGORY_IDS for r in records for c in r["gt_cats"])
n_gt_rel_matched = sum(r["gt_cats"][g] in RELEVANT_CATEGORY_IDS for r in records for _, g, _ in r["matched"])
print("1. GT objects matched, all categories:     ", frac(n_gt_matched, n_gt))
print("   GT objects matched, 19 rule-relevant:     ", frac(n_gt_rel_matched, n_gt_rel))
summary.append(("1. GT matched (all)", 100 * n_gt_matched / n_gt, 33.1, ""))
summary.append(("1. GT matched (19 rule-relevant)", 100 * n_gt_rel_matched / n_gt_rel, None, ""))

n_removed_img = sum(r["num_removed"] > 0 for r in records)
n_added_img = sum(r["num_added"] > 0 for r in records)
n_changed_img = sum(r["num_removed"] > 0 or r["num_added"] > 0 for r in records)
print("2. images with >= 1 removed detection:    ", frac(n_removed_img, n_images))
print("3. images with >= 1 added GT object:      ", frac(n_added_img, n_images))
print("4. images with a changed object set:      ", frac(n_changed_img, n_images))
print("   images unchanged:                       ", frac(n_images - n_changed_img, n_images))
summary.append(("2. images with removal", 100 * n_removed_img / n_images, 55.6, ""))
summary.append(("3. images with addition", 100 * n_added_img / n_images, 75.0, ""))
summary.append(("4. images changed", 100 * n_changed_img / n_images, 83.5, ""))
summary.append(("4. images unchanged", 100 * (n_images - n_changed_img) / n_images, None, ""))

In [ ]:
n_removed = sum(r["num_removed"] for r in records)
n_added = sum(r["num_added"] for r in records)
print(f"6. mean removed per image: {n_removed} / {n_images} = {n_removed / n_images:.3f}")
print(f"   mean added per image:   {n_added} / {n_images} = {n_added / n_images:.3f}")
summary.append(("6. mean removed per image", n_removed / n_images, 1.04, "value, not %"))
summary.append(("6. mean added per image", n_added / n_images, 2.09, "value, not %"))

## Object size


In [ ]:
PERSON_ID = category_id_by_name["person"]

obj79_rows = []
for i, r in enumerate(records):
    # matched objects: kept_emb holds the retained detections in ascending prediction index
    removed = set(r["unmatched_pred"])
    kept_index = {p: k for k, p in enumerate(p for p in range(len(r["stored_boxes"])) if p not in removed)}
    for p, g, _ in r["matched"]:
        c = r["gt_cats"][g]
        if c == PERSON_ID:
            continue
        dim = name_to_dim[category_id_to_name[c]]
        obj79_rows.append({"kind": "matched", "bucket": size_bucket(r["gt_boxes"][g]),
                           "value": r["kept_emb"][kept_index[p], dim].item(), "unchanged": None})
    # added objects: added_boxes are the GT boxes in 224x224 space
    for e, c, box in zip(r["added_emb"], r["added_ids"], r["added_boxes"]):
        if c == PERSON_ID:
            continue
        dim = name_to_dim[category_id_to_name[c]]
        obj79_rows.append({"kind": "added", "bucket": size_bucket(box), "value": e[dim].item(),
                           "unchanged": intervened_aggregated[i, dim].item() == baseline_aggregated[i, dim].item()})
obj79 = pd.DataFrame(obj79_rows)

gt79 = gt_df[~gt_df["person"]]
added79 = obj79[obj79["kind"] == "added"]
matched79 = obj79[obj79["kind"] == "matched"]
assert len(matched79) == int(gt79["matched"].sum())

def count_row(name, pairs):
    # pairs: list of (numerator, denominator) for the three buckets and the total
    return {"quantity": name, **{b: frac(n, d) if d else f"{n} / 0" for b, (n, d) in zip(SIZE_BUCKETS + ["total"], pairs)}}

def per_bucket(df, num_fn):
    return [(num_fn(df[df["bucket"] == b]), len(df[df["bucket"] == b])) for b in SIZE_BUCKETS] + [(num_fn(df), len(df))]

table12 = pd.DataFrame([
    count_row("share of GT objects", [(int((gt79["bucket"] == b).sum()), len(gt79)) for b in SIZE_BUCKETS] + [(len(gt79), len(gt79))]),
    count_row("recall (matched)", per_bucket(gt79, lambda d: int(d["matched"].sum()))),
    count_row("share of added objects", [(int((added79["bucket"] == b).sum()), len(added79)) for b in SIZE_BUCKETS] + [(len(added79), len(added79))]),
    count_row("own concept > 0, added", per_bucket(added79, lambda d: int((d["value"] > 0).sum()))),
    count_row("own concept > 0, matched", per_bucket(matched79, lambda d: int((d["value"] > 0).sum()))),
    count_row("aggregated value unchanged, added", per_bucket(added79, lambda d: int(d["unchanged"].sum()))),
])
print("79 categories excl. person, columns = GT box size bucket")
print(table12.to_string(index=False))

## Reasons for missed objects


In [ ]:
REASON_GROUPS = {
    "removed by size filter": "1. size filter",
    "score in [0.2, 0.5) (default 0.5 threshold)": "2. score in [0.2, 0.5)",
    "cut by top-k (7)": "3. top-k",
    "correct class, IoU in [0.1, 0.5)": "4. localization",
    "wrong class only, IoU in [0.1, 0.5)": "4. localization",
    "no detection (IoU < 0.1)": "5. not detected",
    "below score 0.2": "extra: score < 0.2 (IoU >= 0.5)",
    "suppressed by NMS": "extra: NMS",
    "retained, matched to another GT": "extra: retained, matched to another GT",
}
GROUP_ORDER = ["1. size filter", "2. score in [0.2, 0.5)", "3. top-k", "4. localization", "5. not detected",
               "extra: score < 0.2 (IoU >= 0.5)", "extra: NMS", "extra: retained, matched to another GT"]
miss13 = miss_b.assign(group=miss_b["reason"].map(REASON_GROUPS))
assert miss13["group"].notna().all()

populations = {"(a) 80 categories": miss13, "(b) 79 excl. person": miss13[miss13["category"] != "person"]}
for b in SIZE_BUCKETS:
    pop_b = populations["(b) 79 excl. person"]
    populations[f"(b) {b}"] = pop_b[pop_b["bucket"] == b]
populations["(b) >= 1 %"] = pop_b[pop_b["bucket"] != "< 1 %"]

table13 = {}
for name, df in populations.items():
    counts = df["group"].value_counts().reindex(GROUP_ORDER).fillna(0).astype(int)
    col = {g: f"{n} ({100 * n / len(df):.1f} %)" for g, n in counts.items()}
    ocb = counts[["1. size filter", "2. score in [0.2, 0.5)", "3. top-k"]].sum()
    col["sum 1-3 (OCB settings)"] = f"{ocb} ({100 * ocb / len(df):.1f} %)"
    mask_rcnn = counts[["5. not detected", "extra: score < 0.2 (IoU >= 0.5)"]].sum()
    col["sum Mask R-CNN (5 + score < 0.2)"] = f"{mask_rcnn} ({100 * mask_rcnn / len(df):.1f} %)"
    other = counts[["extra: NMS", "extra: retained, matched to another GT"]].sum()
    col["sum other (NMS + matching conflict)"] = f"{other} ({100 * other / len(df):.1f} %)"
    col["total unmatched GT"] = f"{len(df)} (100.0 %)"
    table13[name] = col
print(pd.DataFrame(table13).to_string())